In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb


def rmse(a, p):
    return np.sqrt(np.mean((a - p) ** 2))


panel = pd.read_parquet("Panel.parquet")

HAR  = ["RV_daily", "RV_weekly", "RV_monthly"]
HARX = HAR + ["CDS_level", "CDS_change_5d", "VIX", "Yield_slope", "BAA10Y"]

RF_PARAMS  = dict(n_estimators=300, max_features=0.5, random_state=42, n_jobs=-1)
XGB_PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05,
                  subsample=0.8, random_state=42, verbosity=0)

print(f"Loaded {len(panel):,} rows, {panel['Firm'].nunique()} firms")

Loaded 211,196 rows, 56 firms


In [2]:
# ML-to-HAR RMSE ratio for 2022, per firm and horizon.
rows = []
for tgt, hlabel in [("Target_h1", "1-day"), ("Target_h5", "1-week"), ("Target_h22", "1-month")]:
    d  = panel.dropna(subset=HARX + [tgt]).copy()
    tr = d[d["Date"].dt.year <  2022]
    te = d[d["Date"].dt.year == 2022]
    ytr = tr[tgt].values
    mh = LinearRegression().fit(tr[HAR].values,  ytr)
    rf = RandomForestRegressor(**RF_PARAMS).fit(tr[HARX].values, ytr)
    xg = xgb.XGBRegressor(**XGB_PARAMS).fit(tr[HARX].values, ytr)
    for firm, g in te.groupby("Firm"):
        a  = g[tgt].values
        ph = np.clip(mh.predict(g[HAR].values),  0, None)
        pr = np.clip(rf.predict(g[HARX].values), 0, None)
        px = np.clip(xg.predict(g[HARX].values), 0, None)
        har_r = rmse(a, ph)
        rows.append({"Horizon": hlabel, "Firm": firm, "Sector": g["Sector"].iloc[0],
                     "HAR_RMSE": har_r,
                     "RF_ratio":  rmse(a, pr) / har_r if har_r > 0 else np.nan,
                     "XGB_ratio": rmse(a, px) / har_r if har_r > 0 else np.nan})

res = pd.DataFrame(rows)
res.to_csv("Decomp_2022.csv", index=False)
print(f"Decomposition complete: {len(res)} firm-horizon rows")

Decomposition complete: 165 firm-horizon rows


In [3]:
for h in ["1-day", "1-week", "1-month"]:
    sub = res[res["Horizon"] == h].copy()
    print(f"\n{h}  RF ratio (ML RMSE / HAR RMSE) in 2022")
    print(f"  firms with RF ratio > 2x HAR: {(sub['RF_ratio'] > 2).sum()} of {len(sub)}")
    print(f"  firms with RF ratio > 5x HAR: {(sub['RF_ratio'] > 5).sum()} of {len(sub)}")
    print(f"  median RF ratio: {sub['RF_ratio'].median():.2f}  |  mean: {sub['RF_ratio'].mean():.2f}")
    print("  worst 5 firms (RF):")
    print(sub.nlargest(5, "RF_ratio")[["Firm", "Sector", "RF_ratio", "XGB_ratio"]].to_string(index=False))
    sub["excess"] = (sub["RF_ratio"] - 1).clip(lower=0)
    top5 = sub.nlargest(5, "excess")["excess"].sum() / sub["excess"].sum() if sub["excess"].sum() > 0 else np.nan
    print(f"  share of total excess RF error from worst 5 firms: {top5:.1%}")


1-day  RF ratio (ML RMSE / HAR RMSE) in 2022
  firms with RF ratio > 2x HAR: 8 of 55
  firms with RF ratio > 5x HAR: 1 of 55
  median RF ratio: 1.43  |  mean: 1.66
  worst 5 firms (RF):
Firm Sector  RF_ratio  XGB_ratio
 RCL    XLY  5.340522   6.508652
 CCL    XLY  3.808027   4.335258
 MGM    XLY  3.226304   4.106355
 JNJ    XLV  2.668278   1.497288
 MCD    XLY  2.317387   1.321568
  share of total excess RF error from worst 5 firms: 34.0%

1-week  RF ratio (ML RMSE / HAR RMSE) in 2022
  firms with RF ratio > 2x HAR: 33 of 55
  firms with RF ratio > 5x HAR: 6 of 55
  median RF ratio: 2.37  |  mean: 2.86
  worst 5 firms (RF):
Firm Sector  RF_ratio  XGB_ratio
 MGM    XLY 11.294366   9.235512
 RCL    XLY  9.921876   9.162683
  BA    XLI  9.455648   7.576244
 CCL    XLY  6.442978   6.022257
 STX    XLK  5.756675   7.759940
  share of total excess RF error from worst 5 firms: 36.9%

1-month  RF ratio (ML RMSE / HAR RMSE) in 2022
  firms with RF ratio > 2x HAR: 55 of 55
  firms with RF ratio